# Projeto de Controladores

In [144]:
# Bibliotecas utilizadas
import matplotlib.pyplot as plt
import sympy as sp
from sympy import symbols, I, Abs, re, im, Poly
from sympy.physics.control import *
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath
from collections import Counter
#import plotly.graph_objects as go

In [145]:
# Símbolos utilizados
s, k = sp.symbols('s k')

## Funções auxiliares

In [146]:
def polinomio(entrada):
    '''
    Função que transforma coeficientes em polinômios
    '''
    coeficientes = entrada.strip().split()  # Transforma a entrada em uma lista de strings e remove espaços extras
    coeficientes.reverse() # Inverte a lista
    lista_polinomio = []

    # Percorre de trás para frente para começar pelo maior expoente
    for exp in range(len(coeficientes) - 1, -1, -1):
        coef = coeficientes[exp]

        # Pula coeficientes que são zero
        if coef == "0":
            continue

        # Formata o termo dependendo do expoente
        if exp > 1:
            termo = f"{coef}*s**{exp}"
        elif exp == 1:
            termo = f"{coef}*s"
        else:
            termo = f"{coef}"

        lista_polinomio.append(termo)

    return " + ".join(lista_polinomio)


In [147]:
def definir_controlador():
    PI = False
    PD = False
    PID = False
    controlador = ""

    while(controlador != "PI" and controlador != "PD" and controlador != "PID"):
        controlador = input("Forneça o tipo de controlador").upper()
        if(controlador == "PI"):
            PI = True
            print("Controlador PI")
        elif(controlador == "PD"):
            PD = True
            print("Controlador PD")
        elif(controlador == "PID"):
            PID = True
            print("Controlador PID")
    return PI,PD,PID


In [148]:
def definir_problema():
    metodo_1 = False
    metodo_2 = False
    metodo_3 = False
    print("Digite 1 para Mp e Epsilon.")
    print("Digite 2 para fator de amortecimento e frequência natural de amortecimento.")
    print("Digite 3 para localização de polos de malha")
    especificacoes = 0
    while(especificacoes < 1 or especificacoes > 3):
        especificacoes = int(input("Digite o número: "))
        if(especificacoes == 1):
            metodo_1 = True
        elif(especificacoes == 2):
            metodo_2 = True
        elif(especificacoes == 3):
            metodo_3 = True
    return metodo_1, metodo_2, metodo_3


In [149]:
def definir_polos_dominantes(metodo1, metodo2, metodo3):
    if(metodo1):
        Mp = float(input("Digite o Mp máximo (%): "))
        print(f"Mp = {Mp}%")
        t_acomodacao = float(input("Digite o tempo de acomodação (s):"))
        tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        while(tol != 2 and tol != 5):
            tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        
        print(f"Acomodação {t_acomodacao}s ({tol}%)")

        # Cálculo do epsilon e frequência natural
        epsilon = math.sqrt(math.log(Mp/100)**2/(math.pi**2 + math.log(Mp/100)**2))
        if(tol == 2):
            w_n = 4/(t_acomodacao*epsilon)
        elif(tol == 5):
            w_n = 3/(t_acomodacao*epsilon)

        w_d = w_n*math.sqrt(1 - epsilon**2)
        sigma = epsilon*w_n

        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d  * I

    if(metodo2):
        epsilon = float(input("Digite o valor do epsilon: "))
        w_n = float(input("Digite o valor da frequência natural de oscilação: "))

        w_d = w_n*math.sqrt(1 - epsilon**2)
        sigma = epsilon*w_n
        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d * I
    if(metodo3):
        parte_real = float(input("Digite a parte real do polo: "))
        parte_imaginaria = float(input("Digite a parte imaginária do polo: "))
        w_d = parte_imaginaria
        sigma = - parte_real
        polo_1 = complex(parte_real, parte_imaginaria)
        polo_2 = complex(parte_real, - parte_imaginaria)

    return polo_1, polo_2, w_d, sigma


In [150]:
def angulo(v):
    """Retorna o argumento de um número complexo em graus."""
    return float(sp.deg(sp.arg(v)))


In [151]:
def calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, pid, pd, pi):
    ponto_si = polo_1

    diferenca_polos = np.empty((0,2))
    diferenca_zeros = np.empty((0,2))

    #print("Diferença entre o ponto e os polos: ")
    for polo in polos:
        diferenca = ponto_si - polo
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[round(modulo,4), fase]]
        diferenca_polos = np.append(diferenca_polos, nova_linha, axis=0)
        #print(f"{polo}: {nova_linha}")
    #print

    #print("Diferença entre o ponto e os zeros: ")
    for zero in zeros:
        diferenca = ponto_si - zero
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[round(modulo,4), fase]]
        diferenca_zeros = np.append(diferenca_zeros, nova_linha, axis=0)
        #print(f"{zero}: {nova_linha}")

    produto_polos = 1
    produto_zeros = 1
    angulo_ponto = 0

    for linha in diferenca_polos:
        produto_polos = produto_polos * linha[0]
        angulo_ponto = angulo_ponto - linha[1]

    for linha in diferenca_zeros:
        produto_zeros = produto_zeros * linha[0]
        angulo_ponto = angulo_ponto + linha[1]

    phi = 180 - angulo_ponto

    while phi < 0:
        phi += 360
        
    phi = phi % 360

    if(pid):
        phi = phi /2

    print(f"Angulo = {phi:.2f}°")

    if phi > 90:
        z_c = abs(sigma) - (w_d / math.tan(math.radians(180 - phi)))
    else:
        z_c = abs(sigma) + (w_d / math.tan(math.radians(phi)))
    print(f"Zero do controlador: {z_c:.2f} + 0*j")

    if(pid):
        Gc = TransferFunction((s + z_c)**2, s, s)
    if(pd):
        Gc = TransferFunction(s + z_c, 1, s)
    if(pi):
        Gc = TransferFunction(s + z_c, s, s)
        
    ft_c = Series(ft, Gc).doit()

    freq = ft_c.eval_frequency(ponto_si)
    M = abs(complex(freq))
    Kc = 1 / M
    print(f"Kc = {Kc}\n")
    
    return z_c, Kc

## Traduzir as especificações de desempenho

In [152]:
PI, PD, PID = definir_controlador()

Controlador PID


In [153]:
metodo_1, metodo_2, metodo_3 = definir_problema()

Digite 1 para Mp e Epsilon.
Digite 2 para fator de amortecimento e frequência natural de amortecimento.
Digite 3 para localização de polos de malha


In [154]:
polo_1, polo_2, w_d, sigma = definir_polos_dominantes(metodo_1, metodo_2, metodo_3)


Mp = 20.0%
Acomodação 5.0s (2%)


In [155]:
print("Forneça os coeficientes da função G(s)...")
numerador_G = polinomio(input("Digite os coeficientes do numerador: "))
denominador_G = polinomio(input("Digite os coeficientes do denominador: "))

numerador_G = sp.parse_expr(numerador_G, local_dict={'s': s})
denominador_G = sp.parse_expr(denominador_G, local_dict={'s': s})

G = TransferFunction(numerador_G, denominador_G, s)
G

Forneça os coeficientes da função G(s)...


TransferFunction(5, s**3 + 12*s**2 + 22*s + 20, s)

In [156]:
print("Forneça os coeficientes da função H(s)...")
numerador_H = polinomio(input("Digite os coeficientes do numerador: "))
denominador_H = polinomio(input("Digite os coeficientes do denominador: "))

numerador_H = sp.parse_expr(numerador_H, local_dict={'s': s})
denominador_H = sp.parse_expr(denominador_H, local_dict={'s': s})

H = TransferFunction(numerador_H, denominador_H, s)
H

Forneça os coeficientes da função H(s)...


TransferFunction(0.4, 1, s)

In [157]:
ft = Series(G, H).doit()
ft

TransferFunction(2.0, s**3 + 12*s**2 + 22*s + 20, s)

In [158]:
zeros, polos = pole_zero_numerical_data(ft)
print(f"Polos: {polos}")
print(f"Zeros: {zeros}")

Polos: [-10.0, (-1-1j), (-1+1j)]
Zeros: []


In [159]:
# Verifica se a função de transferência tem naturalmente um ganho
num = ft.num
den = ft.den

ganho = Poly(num, s).LC() / Poly(den, s).LC()
print(f"Ganho = {ganho}")

Ganho = 2.00000000000000


## PD

In [160]:
if(PD):
    print(f"Polo 1: {polo_1}")
    print(f"Polo 2: {polo_2}\n")
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID, PD, PI)

    Kd = Kc
    Kp = z_c * Kc
        
    Gc = sp.together(Kp +Kd*s)

    print(f"\nKp = {Kp}")
    print(f"Kd = {Kd}")

    print("\nControlador PD:")
    sp.pprint(Gc)
    

## PI

In [161]:
if(PI):
    zeros, polos = pole_zero_numerical_data(ft) # para ao executar novamente não mudar os resultados
    polos.append(0 + 0*I)
    print(f"Polo 1: {polo_1}")
    print(f"Polo 2: {polo_2}")

    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID, PD, PI)

    Kp = Kc
    Ki = z_c * Kc
    
    Gc = sp.together(Kp + Ki/s)

    print(f"\nKp = {Kp}")
    print(f"Ki = {Ki}")

    print("\nControlador PI:")
    sp.pprint(Gc)
    

## PID

In [162]:
if(PID):
    zeros, polos = pole_zero_numerical_data(ft)  # para ao executar novamente não mudar os resultados
    polos.append(0 + 0*I)
    print(f"Polo 1: {polo_1}")
    print(f"Polo 2: {polo_2}")

    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID, PD, PI)

    Kp = 2*Kc*z_c
    Ki = Kc * z_c**2
    Kd = Kc

    Gc = sp.together(Kp + Ki/s + Kd*s)

    print(f"\nKp = {Kp}")
    print(f"Ki = {Ki}")
    print(f"Kd = {Kd}")

    print("\nControlador PID:")
    sp.pprint(Gc)


Polo 1: -0.8 + 1.56158501266494*I
Polo 2: -0.8 - 1.56158501266494*I
Angulo = 51.35°
Zero do controlador: 2.05 + 0*j
Kc = 3.135959580488168


Kp = 12.851145436611981
Ki = 13.165981160959678
Kd = 3.135959580488168

Controlador PID:
                  2                                       
3.13595958048817⋅s  + 12.851145436612⋅s + 13.1659811609597
──────────────────────────────────────────────────────────
                            s                             
